다음은 LangSmith Prompt Hub 에서 프롬프트를 받아서 실행하는 예제입니다.

> 원본: CH02 `03-LangChain-Hub.ipynb`  
> 기준: LangChain 1.x / LangSmith SDK (2026년 9월)

아래 주소에서 공개 프롬프트를 확인할 수 있습니다: https://smith.langchain.com/hub

받아오는 방법은 프롬프트 repo 의 아이디 값을 가져 올 수 있고, commit id 를 붙여서 특정 버전에 대한 프롬프트를 받아올 수도 있습니다.

### 🔄 변경 사항: `langchain.hub` → `langsmith.Client`
- LangChain v1 에서 `from langchain import hub` 는 **`langchain-classic` 패키지로 이동**했습니다 (레거시 경로, `from langchain_classic import hub`).
- LangSmith 공식 문서는 프롬프트 관리에 **LangSmith SDK (`langsmith.Client`)** 사용을 권장합니다.

| 기존 | 최신 권장 |
|---|---|
| `hub.pull("owner/name")` | `client.pull_prompt("owner/name")` |
| `hub.pull("owner/name:커밋해시")` | `client.pull_prompt("owner/name:커밋해시")` |
| `hub.push("owner/name", prompt)` | `client.push_prompt("name", object=prompt)` |

- 내 프롬프트를 올릴 때는 `owner/` 없이 **이름만** 지정하면 API 키가 속한 워크스페이스에 저장됩니다. (책의 `teddynote/...` 를 그대로 쓰면 다른 사람 계정이므로 실패합니다.)
- `.env` 에 `LANGSMITH_API_KEY` 가 필요합니다.

In [ ]:
# %pip install -qU langsmith langchain-core python-dotenv

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # LANGSMITH_API_KEY 로드

## Hub로부터 Prompt 받아오기

In [ ]:
from langsmith import Client

client = Client()

# 가장 최신 버전의 프롬프트를 가져옵니다.
prompt = client.pull_prompt("rlm/rag-prompt")

In [ ]:
# 프롬프트 내용 출력
print(prompt)

In [ ]:
# 특정 버전의 프롬프트를 가져오려면 버전 해시를 지정하세요
prompt = client.pull_prompt("rlm/rag-prompt:50442af1")
prompt

## Prompt Hub 에 자신의 프롬프트 등록

### 🔄 변경 사항
- `from langchain.prompts import ChatPromptTemplate` → `from langchain_core.prompts import ChatPromptTemplate`  
  (`langchain.prompts` 경로는 v1 에서 정리되었습니다. 프롬프트 클래스는 항상 `langchain_core.prompts` 에서 import 합니다.)
- `push_prompt()` 는 설명(`description`), 태그(`tags`), 공개 여부(`is_public`) 등 메타데이터도 함께 지정할 수 있습니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "주어진 내용을 바탕으로 다음 문장을 요약하세요. 답변은 반드시 한글로 작성하세요\n\nCONTEXT: {context}\n\nSUMMARY:"
)
prompt

In [ ]:
# 프롬프트를 허브에 업로드합니다. (owner 없이 이름만 → 내 워크스페이스)
url = client.push_prompt(
    "simple-summary-korean",
    object=prompt,
    description="주어진 context 를 한국어로 요약하는 프롬프트",
    tags=["summary", "korean"],
)
url

업로드에 성공하면 프롬프트의 URL 이 반환됩니다.

`https://smith.langchain.com/prompts/프롬프트명/커밋해시?...`

> 참고: 이미 올린 프롬프트와 **내용이 완전히 같으면** 새 커밋이 생성되지 않고 오류가 날 수 있습니다. 내용을 바꾼 뒤 다시 push 하면 새 버전(커밋)이 쌓입니다.

In [ ]:
# 프롬프트를 허브로부터 가져옵니다. (내 프롬프트는 이름만으로 조회)
pulled_prompt = client.pull_prompt("simple-summary-korean")

In [ ]:
# 프롬프트 내용 출력
print(pulled_prompt)

### (참고) 내 프롬프트 목록 조회

In [ ]:
for p in client.list_prompts(is_public=False).repos:
    print(p.repo_handle, "|", p.description)